### Airline Loyalty Program — Data Exploration

Goal of this notebook: load all four datasets, understand structure,
identify missing values, duplicates, and anomalies before any cleaning.

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

loyalty = pd.read_csv("../data/raw/Customer_Loyalty_History.csv")
flights = pd.read_csv("../data/raw/Customer_Flight_Activity.csv")
calendar = pd.read_csv("../data/raw/Calendar.csv")
dictionary = pd.read_csv("../data/raw/Airline_Loyalty_Data_Dictionary.csv")

print("Loyalty History:", loyalty.shape)
print("Flight Activity:", flights.shape)
print("Calendar:", calendar.shape)
print("Data Dictionary:", dictionary.shape)

Loyalty History: (16737, 16)
Flight Activity: (392936, 8)
Calendar: (2557, 4)
Data Dictionary: (24, 3)


### 1. Data Dictionary — column definitions

In [3]:
print(dictionary.to_string())

                       Table                        Field                                                                            Description
0   Customer Flight Activity               Loyalty Number                                                       Customer's unique loyalty number
1                        NaN                         Year                                                                     Year of the period
2                        NaN                        Month                                                                    Month of the period
3                        NaN                Total Flights                            Sum of Flights Booked (all tickets purchased in the period)
4                        NaN                     Distance                                            Flight distance traveled in the period (km)
5                        NaN           Points Accumulated                                               Loyalty points accumulated

### 2. Customer Loyalty History
Demographics, CLV, enrollment and cancellation info — one row per customer.

In [4]:
loyalty.head()

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN


In [5]:
loyalty.info()

<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loyalty Number      16737 non-null  int64  
 1   Country             16737 non-null  str    
 2   Province            16737 non-null  str    
 3   City                16737 non-null  str    
 4   Postal Code         16737 non-null  str    
 5   Gender              16737 non-null  str    
 6   Education           16737 non-null  str    
 7   Salary              12499 non-null  float64
 8   Marital Status      16737 non-null  str    
 9   Loyalty Card        16737 non-null  str    
 10  CLV                 16737 non-null  float64
 11  Enrollment Type     16737 non-null  str    
 12  Enrollment Year     16737 non-null  int64  
 13  Enrollment Month    16737 non-null  int64  
 14  Cancellation Year   2067 non-null   float64
 15  Cancellation Month  2067 non-null   float64
dtypes: float64(4), 

In [6]:
loyalty.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Loyalty Number,16737.0,NaN,NaN,NaN,549735.880445,258912.132453,100018.0,326603.0,550434.0,772019.0,999986.0
Country,16737,1,Canada,16737,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Province,16737,11,Ontario,5404,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,16737,29,Toronto,3351,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Postal Code,16737,55,V6E 3D9,911,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,16737,2,Female,8410,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Education,16737,5,Bachelor,10475,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Salary,12499.0,NaN,NaN,NaN,79245.609409,35008.297285,-58486.0,59246.5,73455.0,88517.5,407228.0
Marital Status,16737,3,Married,9735,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loyalty Card,16737,3,Star,7637,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing values check

In [7]:
missing = loyalty.isnull().sum()
missing_pct = (missing / len(loyalty)) * 100
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct.round(2)}).sort_values('missing_count', ascending=False)

,missing_count,missing_pct
Cancellation Month,14670,87.65
Cancellation Year,14670,87.65
Salary,4238,25.32
Loyalty Number,0,0.00
City,0,0.00
Postal Code,0,0.00
Province,0,0.00
Country,0,0.00
Education,0,0.00
Gender,0,0.00


### Duplicate Loyalty Numbers check

In [8]:
print("Total rows:", len(loyalty))
print("Unique Loyalty Numbers:", loyalty['Loyalty Number'].nunique())
print("Duplicate rows:", loyalty.duplicated().sum())

Total rows: 16737
Unique Loyalty Numbers: 16737
Duplicate rows: 0


### Categorical columns — unique values

In [9]:
cat_cols = ['Country', 'Province', 'Gender', 'Education', 'Marital Status', 'Loyalty Card', 'Enrollment Type']
for col in cat_cols:
    print(f"\n{col}:")
    print(loyalty[col].value_counts(dropna=False))


Country:
Country
Canada    16737
Name: count, dtype: int64

Province:
Province
Ontario                 5404
British Columbia        4409
Quebec                  3300
Alberta                  969
Manitoba                 658
New Brunswick            636
Nova Scotia              518
Saskatchewan             409
Newfoundland             258
Yukon                    110
Prince Edward Island      66
Name: count, dtype: int64

Gender:
Gender
Female    8410
Male      8327
Name: count, dtype: int64

Education:
Education
Bachelor                10475
College                  4238
High School or Below      782
Doctor                    734
Master                    508
Name: count, dtype: int64

Marital Status:
Marital Status
Married     9735
Single      4484
Divorced    2518
Name: count, dtype: int64

Loyalty Card:
Loyalty Card
Star      7637
Nova      5671
Aurora    3429
Name: count, dtype: int64

Enrollment Type:
Enrollment Type
Standard          15766
2018 Promotion      971
Name: count, dt

### Enrollment and Cancellation year/month ranges

In [10]:
print("Enrollment Year range:", loyalty['Enrollment Year'].min(), "-", loyalty['Enrollment Year'].max())
print("Enrollment Month range:", loyalty['Enrollment Month'].min(), "-", loyalty['Enrollment Month'].max())
print("\nCancellation Year value counts:")
print(loyalty['Cancellation Year'].value_counts(dropna=False))
print("\nNumber of customers who cancelled:", loyalty['Cancellation Year'].notna().sum())

Enrollment Year range: 2012 - 2018
Enrollment Month range: 1 - 12

Cancellation Year value counts:
Cancellation Year
NaN       14670
2018.0      645
2017.0      506
2016.0      427
2015.0      265
2014.0      181
2013.0       43
Name: count, dtype: int64

Number of customers who cancelled: 2067


### CLV and Salary — sanity checks (negatives, extreme outliers)

In [11]:
print("CLV - min:", loyalty['CLV'].min(), "max:", loyalty['CLV'].max())
print("Salary - min:", loyalty['Salary'].min(), "max:", loyalty['Salary'].max())
print("\nNegative Salary count:", (loyalty['Salary'] < 0).sum())
print("Negative CLV count:", (loyalty['CLV'] < 0).sum())

CLV - min: 1898.01 max: 83325.38
Salary - min: -58486.0 max: 407228.0

Negative Salary count: 20
Negative CLV count: 0


### 3. Customer Flight Activity
Monthly records per customer — flights, distance, points.

In [12]:
flights.head()

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100590,2018,6,12,15276,22914.0,0,0
1,100590,2018,7,12,9168,13752.0,0,0
2,100590,2018,5,4,6504,9756.0,0,0
3,100590,2018,10,0,0,0.0,512,92
4,100590,2018,2,0,0,0.0,0,0


In [13]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 392936 entries, 0 to 392935
Data columns (total 8 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Loyalty Number               392936 non-null  int64  
 1   Year                         392936 non-null  int64  
 2   Month                        392936 non-null  int64  
 3   Total Flights                392936 non-null  int64  
 4   Distance                     392936 non-null  int64  
 5   Points Accumulated           392936 non-null  float64
 6   Points Redeemed              392936 non-null  int64  
 7   Dollar Cost Points Redeemed  392936 non-null  int64  
dtypes: float64(1), int64(7)
memory usage: 24.0 MB


In [14]:
flights.describe().T

,count,mean,std,min,25%,50%,75%,max
Loyalty Number,392936.0,550527.519034,258604.580187,100018.0,327688.0,551833.0,772194.0,999986.0
Year,392936.0,2017.513661,0.499814,2017.0,2017.0,2018.0,2018.0,2018.0
Month,392936.0,6.513661,3.445428,1.0,4.0,7.0,10.0,12.0
Total Flights,392936.0,1.294888,1.962675,0.0,0.0,0.0,2.0,28.0
Distance,392936.0,1941.440201,3239.975889,0.0,0.0,0.0,3018.0,67284.0
Points Accumulated,392936.0,2027.172345,3872.139841,0.0,0.0,0.0,3039.0,100926.0
Points Redeemed,392936.0,31.304263,126.653775,0.0,0.0,0.0,0.0,876.0
Dollar Cost Points Redeemed,392936.0,5.635661,22.801167,0.0,0.0,0.0,0.0,158.0


### Missing values and duplicates check

In [15]:
print(flights.isnull().sum())
print("\nDuplicate rows:", flights.duplicated().sum())
print("\nUnique Loyalty Numbers in flights:", flights['Loyalty Number'].nunique())
print("Unique Loyalty Numbers in loyalty history:", loyalty['Loyalty Number'].nunique())

Loyalty Number                 0
Year                           0
Month                          0
Total Flights                  0
Distance                       0
Points Accumulated             0
Points Redeemed                0
Dollar Cost Points Redeemed    0
dtype: int64

Duplicate rows: 1922

Unique Loyalty Numbers in flights: 16737
Unique Loyalty Numbers in loyalty history: 16737


### Year/Month range and row count per customer

In [16]:
print("Year range:", flights['Year'].min(), "-", flights['Year'].max())
print("Month range:", flights['Month'].min(), "-", flights['Month'].max())
print("\nRows per customer - describe:")
print(flights.groupby('Loyalty Number').size().describe())

Year range: 2017 - 2018
Month range: 1 - 12

Rows per customer - describe:
count    16737.000000
mean        23.477087
std          3.876017
min         11.000000
25%         24.000000
50%         24.000000
75%         24.000000
max         72.000000
dtype: float64


### Anomaly checks: negative values, zero flights but nonzero distance/points, etc.

In [17]:
print("Negative Total Flights:", (flights['Total Flights'] < 0).sum())
print("Negative Distance:", (flights['Distance'] < 0).sum())
print("Negative Points Accumulated:", (flights['Points Accumulated'] < 0).sum())
print("Negative Points Redeemed:", (flights['Points Redeemed'] < 0).sum())

print("\nRows with 0 flights but distance > 0:", ((flights['Total Flights'] == 0) & (flights['Distance'] > 0)).sum())
print("Rows with 0 flights but points accumulated > 0:", ((flights['Total Flights'] == 0) & (flights['Points Accumulated'] > 0)).sum())
print("Rows with Points Redeemed > Points Accumulated (same month):", (flights['Points Redeemed'] > flights['Points Accumulated']).sum())

Negative Total Flights: 0
Negative Distance: 0
Negative Points Accumulated: 0
Negative Points Redeemed: 0

Rows with 0 flights but distance > 0: 0
Rows with 0 flights but points accumulated > 0: 0
Rows with Points Redeemed > Points Accumulated (same month): 3075


### 4. Calendar
Date dimension mapping days to month/quarter/year starts.

In [18]:
calendar.head()

,Date,Start of Year,Start of Quarter,Start of Month
0,2012-01-01,2012-01-01,2012-01-01,2012-01-01
1,2012-01-02,2012-01-01,2012-01-01,2012-01-01
2,2012-01-03,2012-01-01,2012-01-01,2012-01-01
3,2012-01-04,2012-01-01,2012-01-01,2012-01-01
4,2012-01-05,2012-01-01,2012-01-01,2012-01-01


In [19]:
calendar.info()
print("\nDate range:", calendar['Date'].min(), "-", calendar['Date'].max())

<class 'pandas.DataFrame'>
RangeIndex: 2557 entries, 0 to 2556
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Date              2557 non-null   str  
 1   Start of Year     2557 non-null   str  
 2   Start of Quarter  2557 non-null   str  
 3   Start of Month    2557 non-null   str  
dtypes: str(4)
memory usage: 179.9 KB

Date range: 2012-01-01 - 2018-12-31


### 5. Cross-table consistency checks

In [20]:
loyalty_ids = set(loyalty['Loyalty Number'])
flight_ids = set(flights['Loyalty Number'])

print("Customers in loyalty history but NOT in flight activity:", len(loyalty_ids - flight_ids))
print("Customers in flight activity but NOT in loyalty history:", len(flight_ids - loyalty_ids))

Customers in loyalty history but NOT in flight activity: 0
Customers in flight activity but NOT in loyalty history: 0


### 6. Final sanity checks before cleaning

In [21]:
# Check: is 2018 a partial year in flight activity?
print(flights.groupby('Year')['Month'].unique())

Year
2017    [7, 6, 8, 12, 3, 5, 9, 1, 10, 4, 11, 2]
2018    [6, 7, 5, 10, 2, 4, 3, 8, 9, 11, 12, 1]
Name: Month, dtype: object


In [22]:
# Check: does anyone cancel BEFORE they enrolled? (should be 0)
cancelled = loyalty[loyalty['Cancellation Year'].notna()].copy()
invalid = cancelled[
    (cancelled['Cancellation Year'] < cancelled['Enrollment Year']) |
    ((cancelled['Cancellation Year'] == cancelled['Enrollment Year']) & 
     (cancelled['Cancellation Month'] < cancelled['Enrollment Month']))
]
print("Customers who 'cancelled' before they enrolled:", len(invalid))

Customers who 'cancelled' before they enrolled: 0
